In [46]:
from pathlib import Path
from functools import lru_cache
import random

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import accelforge as af
import functools

import os
af.set_n_parallel_jobs(os.cpu_count(), print_message=True)

Using 8 parallel jobs


In [47]:
ARCH = Path("designs/arch.yaml")
WORKLOAD = Path("designs/gpt3_6.7B_kv_cache.yaml")
VALIDATOR = Path("designs/gpt3_175b_kv_cache.yaml")
MAPPING = Path("designs/gpt3_6.7B_kv_cache_mapping.yaml")

In [48]:
TARGET_GENERATED_TOKENS = 128

MAPPER_JOBS = 1

lookaheads_to_test = list(range(4, 132, 4))
token_sweep = list(range(0, 4096 + 128, 512))
output_path = Path("milestone_2_results.csv")

def batch_size_for_lookahead(lookahead: int) -> int:
    # Use one query vector per speculative position under test.
    return max(1, int(lookahead))

af.set_n_parallel_jobs(MAPPER_JOBS)

In [49]:
@functools.cache
def draft_spec(tokens: int):
    spec = af.Spec.from_yaml(
        ARCH,
        WORKLOAD,
        jinja_parse_data = {
            "BATCH_SIZE": 1,
            "N_TOKENS": tokens,
            "N_NEW_TOKENS": 1
        }
    )
    # we should ask Michael about this (should we even fuse this in the first place)
    # it might also be a good enough approximation to just the draft spec for the entire lookahead
    for node in spec.arch.nodes:
        if isinstance(node, af.arch.Memory):
            print(f'Keeping all tensors in {node.name}')
            node.tensors.keep = "All"
            break
    spec.mapper.metrics = af.mapper.Metrics.LATENCY
    result = spec.map_workload_to_arch()
    return result

@functools.cache
def validator_spec(tokens: int, lookahead: int):
    spec = af.Spec.from_yaml(
        ARCH,
        VALIDATOR,
        jinja_parse_data = {
            "BATCH_SIZE": lookahead,
            "N_TOKENS": tokens + lookahead,
            "N_NEW_TOKENS": 1
        }
    )
    for node in spec.arch.nodes:
        if isinstance(node, af.arch.Memory):
            node.tensors.keep = "All"
            break
    spec.mapper.metrics = af.mapper.Metrics.LATENCY
    result = spec.map_workload_to_arch()
    return result


@functools.cache
def true_validator_spec(tokens: int, lookahead: int):
    spec = af.Spec.from_yaml(
        ARCH,
        VALIDATOR,
        jinja_parse_data = {
            "BATCH_SIZE": lookahead,
            "N_TOKENS": tokens + lookahead,
            "N_NEW_TOKENS": lookahead
        }
    )
    for node in spec.arch.nodes:
        if isinstance(node, af.arch.Memory):
            node.tensors.keep = "All"
            break
    spec.mapper.metrics = af.mapper.Metrics.LATENCY
    result = spec.map_workload_to_arch()
    return result

In [50]:
ARCH = Path("designs/arch.yaml")
WORKLOAD = Path("designs/gpt3_6.7B_kv_cache.yaml")
VALIDATOR = Path("designs/gpt3_175b_kv_cache.yaml")
MAPPING = Path("designs/gpt3_6.7B_kv_cache_mapping.yaml")

BATCH_SIZE = 1
N_TOKENS = 8192
N_NEW_TOKENS = 1
MAPPER_JOBS = 1

def evaluate_workload(tokens: int, lookahead: int):
    total_energy = 0.0
    total_latency = 0.0
    for i in range(lookahead):
        print(f"Currently validating {i}")
        result = draft_spec(tokens + i)
        total_energy += float(result.energy())
        total_latency += float(result.latency())

    print("on Validator")
    result = validator_spec(tokens, lookahead)
    total_energy += float(result.energy())
    total_latency += float(result.latency())

    return total_energy, total_latency

def default_kv_cached(tokens: int, lookahead: int):
    total_energy = 0.0
    total_latency = 0.0
    for i in range(lookahead):
        print(f"Iteration {i}")
        result = validator_spec(tokens + i, 1)
        total_energy += float(result.energy())
        total_latency += float(result.latency())

    return total_energy, total_latency

In [51]:

def simulate_spec_round(gamma: int, alpha: float):
    for i in range(gamma):
        if random.random() >= alpha:
            return i + 1 
    return gamma + 1   

def speculative_decoding(context_len: int, generate_tokens: int,
                         gamma: int, alpha: float, n_sims):
    total_energy_accum, total_latency_accum = 0.0, 0.0
    tokens_generated = 0
    pos = context_len
    sim_energy, sim_latency = 0.0, 0.0

    while tokens_generated < generate_tokens:
        remaining = generate_tokens - tokens_generated
        effective_gamma = min(gamma, remaining)

        for i in range(effective_gamma):
            result = draft_spec(context_len + i)
            e, l = result.energy(), result.latency()
            sim_energy += e
            sim_latency += l

        result = true_validator_spec(context_len, gamma)
        e, l = result.energy(), result.latency()
        sim_energy += e
        sim_latency += l

        accepted = min(simulate_spec_round(effective_gamma, alpha), remaining)
        tokens_generated += accepted
        pos += accepted

    total_energy_accum += sim_energy
    total_latency_accum += sim_latency

    return total_energy_accum, total_latency_accum


In [ ]:
context_lengths = list(range(128, 129, 128))
gammas = [3, 6, 9]
results = []

for ctx in context_lengths:
    print(f"Context={ctx}: running baseline...")
    base_energy, base_latency = default_kv_cached(ctx, TARGET_GENERATED_TOKENS)

    for gamma in gammas:
        print(f"  gamma={gamma}")
        spec_energy, spec_latency = speculative_decoding(
            ctx, 8, gamma, ALPHA, N_SIMULATIONS
        )
        results.append({
            "context_len": ctx,
            "gamma": gamma,
            "alpha": ALPHA,
            "base_energy": base_energy,
            "base_latency": base_latency,
            "spec_energy": spec_energy,
            "spec_latency": spec_latency,
            "latency_speedup": base_latency / spec_latency,
            "energy_ratio": spec_energy / base_energy,
        })

df = pd.DataFrame(results)
df.to_csv("spec_decoding_results.csv", index=False)
print("Saved to spec_decoding_results.csv")
df

In [ ]:
context_lengths = list(range(128, 1024, 128))
gammas = [3, 6, 9]
results = []

for ctx in context_lengths:
    print(f"Context={ctx}: running baseline...")
    base_energy, base_latency = default_kv_cached(ctx, TARGET_GENERATED_TOKENS)

    for gamma in gammas:
        print(f"  gamma={gamma}")
        spec_energy, spec_latency = speculative_decoding(
            ctx, 8, gamma, ALPHA, N_SIMULATIONS
        )
        results.append({
            "context_len": ctx,
            "gamma": gamma,
            "alpha": ALPHA,
            "base_energy": base_energy,
            "base_latency": base_latency,
            "spec_energy": spec_energy,
            "spec_latency": spec_latency,
            "latency_speedup": base_latency / spec_latency,
            "energy_ratio": spec_energy / base_energy,
        })

df2 = pd.DataFrame(results)
df2.to_csv("spec_decoding_results2.csv", index=False)
print("Saved to spec_decoding_results2.csv")
df2